# TieOut local planner: LoRA fine-tune on Colab

Same training script as the Kaggle run (`finetune/train_kaggle.py`), larger base model. Clones the public repo, trains on the v2 plan-language data (every target plan validated by `tieout.plan.validate`), evaluates on held-out phrasings, merges, exports a Q4_K_M GGUF and an Ollama Modelfile.

Runtime: GPU (A100 or L4). Run all cells.

In [ ]:
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
OVERRIDES = {"BASE_MODEL": BASE_MODEL, "EPOCHS": 1, "BATCH_SIZE": 8, "GRAD_ACCUM": 2, "MODEL_NAME": "planner4b"}
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /content/tieout && git clone -q -b v2 https://github.com/Madhvansh/tieout /content/tieout && cd /content/tieout && git log --oneline -1

In [ ]:
import json, os, shutil
src = "/content/tieout/finetune"
inp = "/content/ft_input"
shutil.rmtree(inp, ignore_errors=True); os.makedirs(inp)
shutil.copy(f"{src}/data_v2/train.jsonl", inp)
shutil.copy(f"{src}/data_v2/val.jsonl", inp)
shutil.copy(f"{src}/system_prompt.v2.txt", f"{inp}/system_prompt.txt")
json.dump(OVERRIDES, open(f"{inp}/overrides.json", "w"))
print(os.listdir(inp))

In [ ]:
!cd /content && FT_WORK=/content/out FT_INPUT=/content/ft_input FT_SCRATCH=/content/ft_scratch python /content/tieout/finetune/train_kaggle.py 2>&1 | grep -v Warning | tail -60

In [ ]:
import json
m = json.load(open("/content/out/metrics.json"))
print(json.dumps({k: m.get(k) for k in ("base_model", "train", "gguf_file", "gguf_mb", "elapsed_min_total")}, indent=1))
print(json.dumps({k: v for k, v in m.get("eval_hf", {}).items() if k != "sample_failures"}, indent=1))
!ls -la /content/out

In [ ]:
from google.colab import files
for name in ("metrics.json", "Modelfile", "val_predictions.jsonl", "train_log.txt"):
    files.download(f"/content/out/{name}")
files.download(f"/content/out/{m['gguf_file']}")